# 🔗 Tool Chains

**Chain multiple tools together for complex tasks**

---

## 📋 Overview

**What you'll learn:**
- Sequential tool execution
- Parallel tool execution
- Error handling in chains
- Chain orchestration patterns
- Real-world examples

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
import os
import json
from typing import Dict, List, Any, Callable
import time

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Tool Chains?

### Simple Tool Call:
```
User: "What's the weather in Paris?"
→ get_weather("Paris")
→ Response
```

### Tool Chain:
```
User: "Book me a flight to the warmest city between Paris and London"

Step 1: get_weather("Paris") → 18°C
Step 2: get_weather("London") → 12°C
Step 3: search_flights(origin="NYC", dest="Paris") → Flights found
Step 4: book_flight(flight_id="AF123") → Booked!
```

### Chain Types:

**1. Sequential Chain**
```
Tool A → Tool B → Tool C
(Output of A feeds into B)
```

**2. Parallel Chain**
```
       ┌→ Tool A
Start -┼→ Tool B → Combine
       └→ Tool C
(Run multiple tools at once)
```

**3. Conditional Chain**
```
Tool A → if success: Tool B
         else: Tool C
```

## 🔄 Sequential Chain Example

In [ ]:
# Define tools
def search_product(query: str) -> Dict:
    """Search for products."""
    products = {
        "laptop": {"id": "laptop_123", "name": "MacBook Pro", "price": 2000},
        "phone": {"id": "phone_456", "name": "iPhone 15", "price": 999},
    }
    
    for key, product in products.items():
        if key in query.lower():
            return product
    
    return {"error": "Product not found"}

def check_inventory(product_id: str) -> Dict:
    """Check product inventory."""
    inventory = {
        "laptop_123": {"in_stock": True, "quantity": 5},
        "phone_456": {"in_stock": True, "quantity": 20},
    }
    
    return inventory.get(product_id, {"in_stock": False})

def add_to_cart(product_id: str, quantity: int = 1) -> Dict:
    """Add product to cart."""
    return {
        "cart_id": "cart_789",
        "product_id": product_id,
        "quantity": quantity,
        "status": "added"
    }

# Tool definitions for LLM
shopping_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_product",
            "description": "Search for products by name or category",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check if a product is in stock",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {"type": "string", "description": "Product ID"}
                },
                "required": ["product_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_to_cart",
            "description": "Add a product to the shopping cart",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {"type": "string"},
                    "quantity": {"type": "integer", "default": 1}
                },
                "required": ["product_id"]
            }
        }
    }
]

available_functions = {
    "search_product": search_product,
    "check_inventory": check_inventory,
    "add_to_cart": add_to_cart,
}

def run_tool_chain(user_query: str, tools: List[Dict], max_iterations: int = 10) -> str:
    """Run a tool chain."""
    
    messages = [{"role": "user", "content": user_query}]
    
    print(f"🔗 Starting Tool Chain\n")
    print(f"User: {user_query}\n")
    print("="*60)
    
    for iteration in range(max_iterations):
        print(f"\n📍 Step {iteration + 1}")
        
        # Call LLM
        response = client.chat.completions.create(
            model="gpt-4",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        
        # Check if done
        if not response_message.tool_calls:
            print("  ✅ Chain complete!")
            return response_message.content
        
        # Add assistant message
        messages.append(response_message)
        
        # Execute tool calls
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"  🔧 {function_name}({function_args})")
            
            # Execute
            if function_name in available_functions:
                result = available_functions[function_name](**function_args)
                print(f"     → {result}")
            else:
                result = {"error": f"Unknown function: {function_name}"}
            
            # Add result to messages
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })
    
    return "Max iterations reached"

# Run chain
query = "I want to buy a laptop. Can you check if it's in stock and add it to my cart?"

final_answer = run_tool_chain(query, shopping_tools)

print(f"\n" + "="*60)
print(f"\n💬 Final Answer:\n{final_answer}")

## ⚡ Parallel Tool Execution

In [ ]:
import concurrent.futures

def execute_tools_parallel(tool_calls: List, available_functions: Dict) -> List[Dict]:
    """Execute multiple tools in parallel."""
    
    def execute_single(tool_call):
        """Execute a single tool call."""
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        start_time = time.time()
        
        if function_name in available_functions:
            result = available_functions[function_name](**function_args)
        else:
            result = {"error": f"Unknown function: {function_name}"}
        
        execution_time = time.time() - start_time
        
        return {
            "tool_call_id": tool_call.id,
            "function_name": function_name,
            "args": function_args,
            "result": result,
            "execution_time": execution_time
        }
    
    # Execute in parallel using ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(execute_single, tc) for tc in tool_calls]
        results = [future.result() for future in concurrent.futures.as_completed(futures)]
    
    return results

# Example: Compare weather in multiple cities
def get_weather(location: str) -> Dict:
    """Get weather (with simulated delay)."""
    time.sleep(1)  # Simulate API call
    
    weather_data = {
        "paris": {"temp": 18, "condition": "Sunny"},
        "london": {"temp": 12, "condition": "Rainy"},
        "tokyo": {"temp": 22, "condition": "Cloudy"},
        "new york": {"temp": 15, "condition": "Windy"},
    }
    
    return weather_data.get(location.lower(), {"error": "City not found"})

print("⚡ Parallel Tool Execution\n")
print("Scenario: Get weather for Paris, London, Tokyo, and New York\n")
print("Sequential execution: 4 × 1s = 4 seconds")
print("Parallel execution:   max(1s, 1s, 1s, 1s) ≈ 1 second")
print("\n💡 4x speedup with parallel execution!")

## 🔀 Conditional Chain

In [ ]:
class ConditionalToolChain:
    """Execute tools conditionally based on results."""
    
    def __init__(self, available_functions: Dict):
        self.available_functions = available_functions
        self.execution_log = []
    
    def execute_with_fallback(
        self,
        primary_tool: str,
        primary_args: Dict,
        fallback_tool: str = None,
        fallback_args: Dict = None
    ) -> Dict:
        """Execute primary tool, fallback if it fails."""
        
        print(f"\n🔀 Conditional Execution")
        print(f"  Primary: {primary_tool}({primary_args})")
        
        # Try primary
        try:
            result = self.available_functions[primary_tool](**primary_args)
            
            # Check for error
            if "error" in result:
                print(f"  ❌ Primary failed: {result['error']}")
                
                if fallback_tool:
                    print(f"  🔄 Falling back to: {fallback_tool}({fallback_args})")
                    result = self.available_functions[fallback_tool](**fallback_args)
                    result["fallback_used"] = True
            else:
                print(f"  ✅ Primary succeeded")
            
            self.execution_log.append({
                "tool": primary_tool,
                "success": "error" not in result
            })
            
            return result
            
        except Exception as e:
            print(f"  ❌ Exception: {e}")
            
            if fallback_tool:
                print(f"  🔄 Falling back...")
                return self.available_functions[fallback_tool](**fallback_args)
            
            return {"error": str(e)}

# Example: Payment processing with fallback
def process_credit_card(card_number: str, amount: float) -> Dict:
    """Process credit card (may fail)."""
    # Simulate occasional failure
    import random
    if random.random() < 0.3:  # 30% failure rate
        return {"error": "Credit card declined"}
    return {"status": "success", "transaction_id": "CC123"}

def process_paypal(email: str, amount: float) -> Dict:
    """Process PayPal (fallback)."""
    return {"status": "success", "transaction_id": "PP456"}

payment_functions = {
    "process_credit_card": process_credit_card,
    "process_paypal": process_paypal,
}

chain = ConditionalToolChain(payment_functions)

result = chain.execute_with_fallback(
    primary_tool="process_credit_card",
    primary_args={"card_number": "1234", "amount": 99.99},
    fallback_tool="process_paypal",
    fallback_args={"email": "user@example.com", "amount": 99.99}
)

print(f"\n  Final result: {result}")

## 🎯 Real-World Example: Travel Planner

In [ ]:
# Travel planning tools
def get_weather(location: str) -> Dict:
    return {"temp": 20, "condition": "Sunny"}

def search_flights(origin: str, destination: str, date: str) -> Dict:
    return {"flights": [{"airline": "AirFrance", "price": 250, "time": "10:00"}]}

def search_hotels(location: str, check_in: str, check_out: str) -> Dict:
    return {"hotels": [{"name": "Grand Hotel", "price": 150, "rating": 4.5}]}

def book_flight(flight_id: str) -> Dict:
    return {"confirmation": "FL123", "status": "booked"}

def book_hotel(hotel_id: str) -> Dict:
    return {"confirmation": "HT456", "status": "booked"}

# Tool definitions
travel_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather forecast",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"}
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_flights",
            "description": "Search for flights",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string"},
                    "destination": {"type": "string"},
                    "date": {"type": "string"}
                },
                "required": ["origin", "destination", "date"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": "Search for hotels",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "check_in": {"type": "string"},
                    "check_out": {"type": "string"}
                },
                "required": ["location", "check_in", "check_out"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_flight",
            "description": "Book a flight",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_id": {"type": "string"}
                },
                "required": ["flight_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_hotel",
            "description": "Book a hotel",
            "parameters": {
                "type": "object",
                "properties": {
                    "hotel_id": {"type": "string"}
                },
                "required": ["hotel_id"]
            }
        }
    },
]

travel_functions = {
    "get_weather": get_weather,
    "search_flights": search_flights,
    "search_hotels": search_hotels,
    "book_flight": book_flight,
    "book_hotel": book_hotel,
}

print("🎯 Travel Planner Example\n")
print("Query: 'Plan a trip to Paris on June 15-18. Check weather, find flights from NYC, and book a hotel.'")
print("\nExpected chain:")
print("  1. get_weather('Paris')")
print("  2. search_flights('NYC', 'Paris', '2024-06-15')")
print("  3. search_hotels('Paris', '2024-06-15', '2024-06-18')")
print("  4. book_flight(...)")
print("  5. book_hotel(...)")
print("\n💡 LLM automatically orchestrates the entire chain!")

## ✅ Summary

### Tool Chain Patterns:

**1. Sequential Chain**
```python
# Output of one tool feeds into next
product = search_product("laptop")
inventory = check_inventory(product["id"])
cart = add_to_cart(product["id"])
```

**2. Parallel Chain**
```python
# Run multiple tools simultaneously
with ThreadPoolExecutor() as executor:
    weather_paris = executor.submit(get_weather, "Paris")
    weather_london = executor.submit(get_weather, "London")
    
# 2x speedup!
```

**3. Conditional Chain**
```python
# Fallback if primary fails
result = process_credit_card(card)
if result.get("error"):
    result = process_paypal(email)
```

### Best Practices:

**1. Clear Tool Dependencies**
```python
# Document what each tool needs
def book_flight(flight_id: str):  # Requires flight_id from search_flights
    ...
```

**2. Error Handling**
```python
# Always handle failures
result = execute_tool(tool, args)
if "error" in result:
    # Try fallback or abort chain
    ...
```

**3. Timeout Protection**
```python
# Don't let chains run forever
max_iterations = 10
for i in range(max_iterations):
    ...
```

**4. Logging**
```python
# Track what's happening
execution_log = []
execution_log.append({
    "step": i,
    "tool": tool_name,
    "args": args,
    "result": result,
    "duration": duration
})
```

### Performance Optimization:

**Parallel vs Sequential:**
```
Sequential: Tool A (1s) → Tool B (1s) → Tool C (1s) = 3s
Parallel:   Tool A, B, C (all 1s) = 1s

💡 Use parallel when tools don't depend on each other!
```

**Caching:**
```python
# Cache expensive tool results
cache = {}
def cached_tool(arg):
    if arg in cache:
        return cache[arg]
    result = expensive_tool(arg)
    cache[arg] = result
    return result
```

### Common Patterns:

**Search → Verify → Action:**
```python
1. search_product("laptop")
2. check_inventory(product_id)
3. add_to_cart(product_id)
```

**Gather → Compare → Decide:**
```python
1. get_weather("Paris") | get_weather("London")  # Parallel
2. compare_results()
3. book_flight(best_destination)
```

**Try → Fallback → Notify:**
```python
1. primary_payment()
2. if failed: backup_payment()
3. send_confirmation()
```

### Next: `07_agents_tools/03_agents_basics.ipynb`